📊 Clusterización de artículos aplicables a forecast

Este notebook desarrolla la segmentación matemática de los artículos que aplican a forecast, utilizando variables construidas a partir del historial de ventas de los últimos 24 meses.

Procedimiento general

1. Importación del dataset preparado
Carga de la base filtrada y enriquecida proveniente del notebook anterior.

2. Validación de variables
Revisión de columnas, tipos de datos, valores nulos, infinitos y consistencia general.

3. Selección de variables de clusterización
Uso de variables finales asociadas a volumen, frecuencia, intermitencia, recencia, variabilidad, picos, tendencia y estacionalidad.

4. Preparación de la matriz de datos
Separación entre columnas de identificación del artículo y variables numéricas para modelado.

5. Escalamiento de variables
Estandarización de los datos para evitar que variables de mayor magnitud dominen el agrupamiento.

6. Evaluación del número óptimo de clusters
Aplicación del método del codo, Silhouette Score y análisis visual.

7. Aplicación de modelos de clusterización
Prueba inicial con K-Means y posible comparación con otros métodos no supervisados.

8. Visualización de resultados
Uso de PCA y gráficos exploratorios para observar la separación de los grupos.

9. Fotografía matemática por cluster
Análisis de promedios, medianas, dispersión y distribución de artículos por grupo.

10. Interpretación de clusters
Asignación de nombres descriptivos según el comportamiento histórico de demanda.

11. Preparación para forecast por grupo
Uso de los clusters como base para comparar modelos predictivos y seleccionar el algoritmo más adecuado por tipo de demanda.

In [2]:
# ============================================================
# 1. Importación de librerías
# ============================================================
# Se importan las librerías necesarias para manipulación de datos,
# visualización, escalamiento, clusterización y evaluación de clusters.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA

In [4]:
from datetime import datetime

# ============================================================
# Exportar dataset preparado para clusterización
# ============================================================

fecha_hoy = datetime.today().strftime("%Y_%m_%d")

nombre_archivo = f"df_a_clusterizar_variables_cluster_final_{fecha_hoy}.xlsx"

ruta=f"./Datos/{nombre_archivo}"


In [5]:
# ============================================================
# 2. Carga del dataset preparado para clusterización
# ============================================================
# Este archivo proviene del notebook anterior, donde se filtraron
# los artículos que aplican a forecast y se calcularon las variables
# matemáticas de comportamiento histórico de venta.
# ============================================================

df_cluster = pd.read_excel(ruta)

df_cluster.head()

,codigo_articulo,descripcion_completa,grupo_articulo,categoria,subcategoria,tipo,marca_fabricante,unidad_medida_inventario,aplica_forecast,motivo_aplica_forecast,total_unidades_vendidas_24m,conteo_facturas_24m,ratio_venta_reciente_6m,cantidad_meses_pico_24m,meses_desde_ultima_venta,pendiente_tendencia_24m,correlacion_estacional_12m,cv_meses_con_venta_24m
0,US2: QF120A,Interruptor 20 A 1P 120 V 10K QPF2 GFCI 5MA,EQUI.AUTO.Y CONTROL,EQUIPO ELECTRICO & AUTOMATIZACION,CENTROS DE CARGA E INTERRUPTORES,NaN,SIEMENS,UNI,True,Aplica forecast,11225.00,67,0.845612,4,0,15.180870,-0.668349,0.495099
1,W8,"Cinta para husos, Habasit, W-8",FAJAS,BANDAS,CORREAS PLANAS DE TRANSMISION,CORREAS PLANAS DE TRANSMISION,HABASIT ROCUA,CM2,True,Aplica forecast,5975816.38,56,0.964050,4,1,-5326.251196,-0.275883,0.989061
2,RB330-3PLY-30''-SHRD,"Banda de Hule y lona de 30""",FAJAS,CORREAS,REDONDAS,REDONDAS,SHARDA,MT,True,Aplica forecast,2058.42,47,0.507788,2,1,-0.823330,-0.224816,1.285581
3,C3.5-TEH-14.984,RODILLO RETRO PSV/1.20F14.89N.38,POLEAS RODI.Y TAMB.,NaN,NaN,NaN,PRECISION INC,UNI,True,Aplica forecast,572.00,20,1.349650,6,0,1.013913,-0.137333,0.507343
4,RB330-3PLY-24''-SHRD,"Banda de Hule y lona de 24"" 3 lonas",FAJAS,CORREAS,REDONDAS,REDONDAS,SHARDA,MT,True,Aplica forecast,1393.68,46,1.223552,3,0,0.652261,-0.291110,0.661989


In [6]:
# ============================================================
# 3. Validación inicial del dataset
# ============================================================

print("Dimensiones del dataset:")
print(df_cluster.shape)

print("\nColumnas disponibles:")
print(df_cluster.columns.tolist())

print("\nTipos de datos:")
df_cluster.info()

Dimensiones del dataset:
(450, 18)

Columnas disponibles:
['codigo_articulo', 'descripcion_completa', 'grupo_articulo', 'categoria', 'subcategoria', 'tipo', 'marca_fabricante', 'unidad_medida_inventario', 'aplica_forecast', 'motivo_aplica_forecast', 'total_unidades_vendidas_24m', 'conteo_facturas_24m', 'ratio_venta_reciente_6m', 'cantidad_meses_pico_24m', 'meses_desde_ultima_venta', 'pendiente_tendencia_24m', 'correlacion_estacional_12m', 'cv_meses_con_venta_24m']

Tipos de datos:
<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   codigo_articulo              450 non-null    str    
 1   descripcion_completa         449 non-null    str    
 2   grupo_articulo               450 non-null    str    
 3   categoria                    411 non-null    str    
 4   subcategoria                 411 non-null    str    
 5   tipo     